In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import dask.dataframe as dd
import pandas as pd
import numpy as np
from sqlalchemy import select, create_engine
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster
from src.utils.stochastic_models import OrnsteinUhlenbeck
import statsmodels.api as sm

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

CLUSTER_TYPE = "local"
N_WORKERS = 10

engine = create_engine(POSTGRES_URL)

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="2GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="prefect-cluster",
        n_workers=N_WORKERS,
        container="ghcr.io/manning-capital/mc-notebooks:main",
        worker_memory="32GB",
    )

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
# cluster = Cluster(
#     name="prefect-cluster",
#     n_workers=3,
#     container="ghcr.io/manning-capital/mc-notebooks:main",
#     worker_memory="32GB",
# )
# client = cluster.get_client()

In [ ]:
@delayed
def load_pairs_trading_frame_chunk(
    provider_asset_group_ids: list[int],
    start: dt.datetime,
    end: dt.datetime,
    conn_string: str,
) -> pd.DataFrame:
    """
    Load the pairs trading frame for a chunk of provider asset groups.
    Returns only the essential columns needed for cointegration analysis.

    Args:
        provider_asset_group_ids: List of provider asset group IDs to process
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        conn_string: Database connection string

    Returns:
        pandas DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    engine = create_engine(conn_string)

    # Step 1: Generate timeframe
    start_str = start.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
    end_str = end.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")

    time_frame = pd.read_sql(
        select(
            func.generate_series(
                literal_column(start_str),
                literal_column(end_str),
                func.cast(literal("1 minute"), INTERVAL),
            ).label("timestamp")
        ),
        engine,
    )

    # Step 2: Load provider asset group members
    members = pd.read_sql(
        select(
            models.ProviderAssetGroupMember.provider_asset_group_id,
            models.ProviderAssetGroupMember.order,
            models.ProviderAssetGroupMember.provider_id,
            models.ProviderAssetGroupMember.from_asset_id,
            models.ProviderAssetGroupMember.to_asset_id,
        ).where(
            models.ProviderAssetGroupMember.provider_asset_group_id.in_(
                provider_asset_group_ids
            )
        ),
        engine,
    )

    # Step 3: Cross join
    time_frame["key"] = 1
    members["key"] = 1
    full_frame = time_frame.merge(members, on="key").drop(columns=["key"])
    full_frame = full_frame.sort_values("timestamp")

    # Step 4: Load market data
    market_data = pd.read_sql(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.ProviderAssetMarket.from_asset_id,
            models.ProviderAssetMarket.to_asset_id,
            models.ProviderAssetMarket.close,
        )
        .where(models.ProviderAssetMarket.timestamp.between(start, end))
        .order_by(models.ProviderAssetMarket.timestamp),
        engine,
    )

    # Step 5: Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame.sort_values("timestamp"),
        market_data.sort_values("timestamp"),
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Step 6: Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})

    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    engine.dispose()
    return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    conn_string: str,
    n_workers: int = 10,
) -> dd.DataFrame:
    """
    Get the pairs trading frame with only essential columns for cointegration analysis.

    Returns:
        Dask DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Split provider asset groups into chunks
    n_chunks = min(n_workers, len(provider_asset_group_ids))
    group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

    # Create delayed tasks
    delayed_dfs = [
        load_pairs_trading_frame_chunk(chunk.tolist(), start, end, conn_string)
        for chunk in group_chunks
    ]

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)

    return pairs_trading_frame

In [ ]:
# @delayed
# def load_provider_group_members_chunk(
#     group_ids: list[int], conn_string: str
# ) -> pd.DataFrame:
#     """
#     Load provider asset group members for a list of group_ids.
#     """
#     engine = create_engine(conn_string)
#     df = pd.read_sql(
#         select(
#             models.ProviderAssetGroupMember.provider_asset_group_id,
#             models.ProviderAssetGroupMember.order,
#             models.ProviderAssetGroupMember.provider_id,
#             models.ProviderAssetGroupMember.from_asset_id,
#             models.ProviderAssetGroupMember.to_asset_id,
#         ).where(models.ProviderAssetGroupMember.provider_asset_group_id.in_(group_ids)),
#         engine,
#     )
#     engine.dispose()
#     return df


# @delayed
# def load_market_data_chunk(start_time, end_time, conn_string: str) -> pd.DataFrame:
#     """
#     Load a chunk of market data for a time range.
#     """
#     engine = create_engine(conn_string)
#     df = pd.read_sql(
#         select(
#             models.ProviderAssetMarket.timestamp,
#             models.ProviderAssetMarket.provider_id,
#             models.ProviderAssetMarket.from_asset_id,
#             models.ProviderAssetMarket.to_asset_id,
#             models.ProviderAssetMarket.close,
#         )
#         .where(models.ProviderAssetMarket.timestamp.between(start_time, end_time))
#         .order_by(models.ProviderAssetMarket.timestamp),
#         engine,
#         index_col="timestamp",
#     )
#     engine.dispose()
#     return df.reset_index()


# def get_pairs_trading_frame(
#     start: dt.datetime,
#     end: dt.datetime,
#     provider_asset_group_ids: list[int],
#     conn_string: str,
#     n_workers: int = 10,
# ) -> dd.DataFrame:
#     """
#     Get the pairs trading frame for given parameters.

#     This function:
#     1. Generates a complete timeframe (1-minute intervals)
#     2. Loads provider asset group members in parallel
#     3. Creates a cross join to get all timestamp-member combinations
#     4. Loads market data in parallel
#     5. Performs merge_asof to join market prices
#     6. Splits by order and creates pairs (order 1 vs order 2)

#     Args:
#         start: Start datetime (timezone-naive)
#         end: End datetime (timezone-naive)
#         provider_asset_group_ids: List of provider asset group IDs to process
#         conn_string: Database connection string
#         n_workers: Number of parallel workers for loading data

#     Returns:
#         Dask DataFrame with columns:
#             - timestamp (index)
#             - provider_asset_group_id
#             - provider_id_1, from_asset_id_1, to_asset_id_1, close_1
#             - provider_id_2, from_asset_id_2, to_asset_id_2, close_2
#     """
#     # Step 1: Generate timeframe
#     start_str = start.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
#     end_str = end.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")

#     time_frame = dd.read_sql_query(
#         select(
#             select(
#                 func.generate_series(
#                     literal_column(start_str),
#                     literal_column(end_str),
#                     func.cast(literal("1 minute"), INTERVAL),
#                 ).label("timestamp")
#             ).subquery("time_frame")
#         ),
#         conn_string,
#         index_col="timestamp",
#         bytes_per_chunk="512 MiB",
#     )
#     time_frame = time_frame.reset_index()

#     # Step 2: Load provider asset group members (in parallel)
#     n_partitions = min(n_workers, len(provider_asset_group_ids))
#     group_chunks = np.array_split(provider_asset_group_ids, n_partitions)

#     delayed_member_dfs = [
#         load_provider_group_members_chunk(chunk.tolist(), conn_string)
#         for chunk in group_chunks
#     ]

#     meta_members = pd.DataFrame(
#         {
#             "provider_asset_group_id": pd.Series(dtype="int64"),
#             "order": pd.Series(dtype="int64"),
#             "provider_id": pd.Series(dtype="int64"),
#             "from_asset_id": pd.Series(dtype="int64"),
#             "to_asset_id": pd.Series(dtype="int64"),
#         }
#     )

#     provider_asset_group_members = dd.from_delayed(
#         delayed_member_dfs, meta=meta_members
#     )

#     # Step 3: Cross join timeframe with members
#     time_frame["key"] = 1
#     provider_asset_group_members["key"] = 1
#     full_frame = time_frame.merge(provider_asset_group_members, on="key")
#     full_frame = full_frame.drop(columns=["key"])
#     full_frame = full_frame.sort_values(by="timestamp")
#     full_frame = full_frame.set_index("timestamp")

#     # Step 4: Load market data (in parallel)
#     time_chunks = pd.date_range(start, end, periods=n_workers + 1)

#     delayed_market_dfs = [
#         load_market_data_chunk(time_chunks[i], time_chunks[i + 1], conn_string)
#         for i in range(len(time_chunks) - 1)
#     ]

#     meta_market = pd.DataFrame(
#         {
#             "timestamp": pd.Series(dtype="datetime64[ns]"),
#             "provider_id": pd.Series(dtype="int64"),
#             "from_asset_id": pd.Series(dtype="int64"),
#             "to_asset_id": pd.Series(dtype="int64"),
#             "close": pd.Series(dtype="float64"),
#         }
#     )

#     market_data = dd.from_delayed(delayed_market_dfs, meta=meta_market)
#     market_data = market_data.sort_values(by="timestamp")
#     market_data = market_data.set_index("timestamp")

#     # Step 5: Merge_asof to join market prices
#     full_market_frame = dd.merge_asof(
#         full_frame,
#         market_data,
#         left_index=True,
#         right_index=True,
#         by=["provider_id", "from_asset_id", "to_asset_id"],
#     )

#     # Step 6: Split by order and create pairs
#     close_1 = full_market_frame.loc[
#         full_frame["order"] == 1,
#         [
#             "provider_asset_group_id",
#             "provider_id",
#             "from_asset_id",
#             "to_asset_id",
#             "close",
#         ],
#     ]
#     close_2 = full_market_frame.loc[
#         full_frame["order"] == 2,
#         [
#             "provider_asset_group_id",
#             "provider_id",
#             "from_asset_id",
#             "to_asset_id",
#             "close",
#         ],
#     ]

#     pairs_trading_frame = dd.merge(
#         close_1,
#         close_2,
#         on=["timestamp", "provider_asset_group_id"],
#         how="inner",
#         suffixes=("_1", "_2"),
#     )

#     return pairs_trading_frame

In [ ]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=30)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

In [ ]:
max_groups = 250
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    provider_asset_group_ids,
    engine.url.render_as_string(hide_password=False),
    N_WORKERS,
)

In [ ]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)

In [ ]:
cointegration_p_values_computed = cointegration_p_values.compute()
cointegration_p_values_computed

In [ ]:
cointegration_p_values_computed.to_parquet("cointegration_p_values.parquet")

In [ ]:
cointegration_p_values_computed = dd.read_parquet("cointegration_p_values.parquet")

In [ ]:
cointegrated_provider_asset_group_ids = (
    cointegration_p_values_computed.loc[
        cointegration_p_values_computed["p_value"] < 0.001
    ]
    .compute()
    .index.tolist()
)
print(
    f"Cointegrated provider asset group ids (count: {len(cointegrated_provider_asset_group_ids)}): {cointegrated_provider_asset_group_ids}"
)

In [ ]:
def get_cointegrated_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Get the cointegrated stats for a given dataframe.
    """

    # Compute the linear regression.
    X = df["close_1"].to_numpy()
    y = df["close_2"].to_numpy()
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()

    # Get the residuals.
    linear_fit_alpha = results.params[0]
    linear_fit_beta = results.params[1]
    linear_fit_mse = results.mse_total
    linear_fit_r_squared = results.rsquared
    linear_fit_r_squared_adj = results.rsquared_adj
    residuals = results.resid

    # Get the cointegration stats.
    ou_params = OrnsteinUhlenbeck().fit(residuals)

    return pd.Series(
        [
            linear_fit_alpha,
            linear_fit_beta,
            linear_fit_mse,
            linear_fit_r_squared,
            linear_fit_r_squared_adj,
            ou_params.mu,
            ou_params.theta,
            ou_params.sigma,
        ],
        index=[
            "linear_fit_alpha",
            "linear_fit_beta",
            "linear_fit_mse",
            "linear_fit_r_squared",
            "linear_fit_r_squared_adj",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
        ],
        dtype=float,
    )

In [ ]:
cointegrated_pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    cointegrated_provider_asset_group_ids,
    engine.url.render_as_string(hide_password=False),
    N_WORKERS,
)

In [ ]:
cointegrated_pairs_trading_stats = cointegrated_pairs_trading_frame.groupby(
    "provider_asset_group_id"
)[["close_1", "close_2"]].apply(
    lambda df: get_cointegrated_stats(df),
    meta={
        "linear_fit_alpha": pd.Series([], dtype=float),
        "linear_fit_beta": pd.Series([], dtype=float),
        "linear_fit_mse": pd.Series([], dtype=float),
        "linear_fit_r_squared": pd.Series([], dtype=float),
        "linear_fit_r_squared_adj": pd.Series([], dtype=float),
        "ou_mu": pd.Series([], dtype=float),
        "ou_theta": pd.Series([], dtype=float),
        "ou_sigma": pd.Series([], dtype=float),
    },
)

In [ ]:
cointegrated_pairs_trading_stats_computed = cointegrated_pairs_trading_stats.compute()
cointegrated_pairs_trading_stats_computed